# DiabCare AI — Phase 3: Explainability Demo

This notebook demonstrates the `explain_patient()` function that forms the core of the DiabCare AI system. It takes a single patient's raw clinical record and returns:
- **Risk %** — predicted probability of 30-day hospital readmission  
- **Risk category** — Low (<30%) / Moderate (30–60%) / High (>60%)  
- **Top 3 plain-language factors** — SHAP-based, describing which features most influenced this specific score  
- **Follow-up priority** — Low / Medium / High  

> **Disclaimer:** This is a prototype trained on historical US hospital data. AUC ~0.67 is the honest expected performance. SHAP values indicate feature correlation, NOT causation. Risk thresholds are prototype cutoffs, not clinically validated.

In [1]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import joblib
import pandas as pd

from Src.explain import explain_patient
from Src.Preprocessing import load_data

# Load pipeline and data once
pipeline = joblib.load("../DATA/lgbm_pipeline.joblib")
X, y = load_data("../DATA/CleanedDiabetic_data.csv")

print(f"Dataset loaded: {X.shape[0]} patients, {X.shape[1]} features")
print(f"Pipeline loaded: {type(pipeline.named_steps['model']).__name__}")

d:\DiabCare\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset loaded: 101766 patients, 44 features
Pipeline loaded: LGBMClassifier


## Example 1 — Low Risk Patient

In [2]:
# Patient #100: discharge_disposition_id = 11 (expired), so cannot be readmitted
idx = 100
row = X.iloc[[idx]]
true_label = y.iloc[idx]

result = explain_patient(row, pipeline=pipeline)

print(f"True outcome : {'Readmitted <30 days' if true_label == 1 else 'Not readmitted'}")
print(f"Risk         : {result['risk_percent']}%  [{result['risk_category']}]")
print(f"Priority     : {result['follow_up_priority']}")
print(f"")
print(f"Top factors driving this score:")
for i, f in enumerate(result['top_factors'], 1):
    print(f"  {i}. {f['factor']} -- {f['direction']}")

True outcome : Not readmitted
Risk         : 0.7%  [Low]
Priority     : Low

Top factors driving this score:
  1. Patient expired during stay -- decreases risk
  2. Discharged to home -- decreases risk
  3. Number of prior inpatient visits -- decreases risk


d:\DiabCare\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## Example 2 — Moderate Risk Patient

In [3]:
idx = 500
row = X.iloc[[idx]]
true_label = y.iloc[idx]

result = explain_patient(row, pipeline=pipeline)

print(f"True outcome : {'Readmitted <30 days' if true_label == 1 else 'Not readmitted'}")
print(f"Risk         : {result['risk_percent']}%  [{result['risk_category']}]")
print(f"Priority     : {result['follow_up_priority']}")
print(f"")
print(f"Top factors driving this score:")
for i, f in enumerate(result['top_factors'], 1):
    print(f"  {i}. {f['factor']} -- {f['direction']}")

True outcome : Not readmitted
Risk         : 37.5%  [Moderate]
Priority     : Medium

Top factors driving this score:
  1. Number of prior inpatient visits -- decreases risk
  2. Discharged to home -- increases risk
  3. Primary diagnosis code 571 -- decreases risk


d:\DiabCare\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## Example 3 — High Risk Patient (confirmed readmission)

In [4]:
# Find a patient with many prior inpatient visits (higher risk)
high_risk_candidates = X[X["number_inpatient"] >= 4]
idx = high_risk_candidates.index[0]
row = X.loc[[idx]]
true_label = y.loc[idx]

result = explain_patient(row, pipeline=pipeline)

print(f"Patient index: {idx}")
print(f"Prior inpatient visits: {row['number_inpatient'].values[0]}")
print(f"True outcome : {'Readmitted <30 days' if true_label == 1 else 'Not readmitted'}")
print(f"Risk         : {result['risk_percent']}%  [{result['risk_category']}]")
print(f"Priority     : {result['follow_up_priority']}")
print(f"")
print(f"Top factors driving this score:")
for i, f in enumerate(result['top_factors'], 1):
    print(f"  {i}. {f['factor']} -- {f['direction']}")

Patient index: 175
Prior inpatient visits: 6
True outcome : Not readmitted
Risk         : 73.0%  [High]
Priority     : High

Top factors driving this score:
  1. Number of prior inpatient visits -- increases risk
  2. Length of hospital stay -- decreases risk
  3. Discharged to home -- increases risk


d:\DiabCare\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## Clinical Direction Sanity Check

Verifying that SHAP directions align with clinical intuition.

In [5]:
checks = []

# Check 1: High prior inpatient visits vs zero
high_inp = X[X["number_inpatient"] >= 3].iloc[:10]
low_inp  = X[X["number_inpatient"] == 0].iloc[:10]
high_risks = [explain_patient(high_inp.iloc[[i]], pipeline=pipeline)["risk_percent"] for i in range(len(high_inp))]
low_risks  = [explain_patient(low_inp.iloc[[i]], pipeline=pipeline)["risk_percent"]  for i in range(len(low_inp))]
checks.append({
    "Check": "High (>=3) vs zero prior inpatient visits",
    "High group avg risk %": round(sum(high_risks)/len(high_risks), 1),
    "Low group avg risk %": round(sum(low_risks)/len(low_risks), 1),
    "Expected": "High > Low",
    "Result": "PASS" if sum(high_risks)/len(high_risks) > sum(low_risks)/len(low_risks) else "FAIL"
})

# Check 2: Expired discharge (code 11)
expired = X[X["discharge_disposition_id"] == 11].iloc[:10]
if len(expired) > 0:
    exp_risks = [explain_patient(expired.iloc[[i]], pipeline=pipeline)["risk_percent"] for i in range(len(expired))]
    avg_exp = sum(exp_risks)/len(exp_risks)
    checks.append({
        "Check": "Expired discharge (code 11) avg risk",
        "High group avg risk %": round(avg_exp, 1),
        "Low group avg risk %": "N/A",
        "Expected": "< 5% (deceased)",
        "Result": "PASS" if avg_exp < 5 else "REVIEW"
    })

# Check 3: Long vs short stays
long_s  = X[X["time_in_hospital"] >= 10].iloc[:10]
short_s = X[X["time_in_hospital"] <= 2].iloc[:10]
long_r  = [explain_patient(long_s.iloc[[i]],  pipeline=pipeline)["risk_percent"] for i in range(len(long_s))]
short_r = [explain_patient(short_s.iloc[[i]], pipeline=pipeline)["risk_percent"] for i in range(len(short_s))]
checks.append({
    "Check": "Long stay (>=10d) vs short stay (<=2d)",
    "High group avg risk %": round(sum(long_r)/len(long_r), 1),
    "Low group avg risk %": round(sum(short_r)/len(short_r), 1),
    "Expected": "Long > Short",
    "Result": "PASS" if sum(long_r)/len(long_r) > sum(short_r)/len(short_r) else "REVIEW"
})

pd.DataFrame(checks)

d:\DiabCare\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
d:\DiabCare\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
d:\DiabCare\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
d:\DiabCare\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
d:\DiabCare\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
d:\DiabCare\.venv\Lib\site-pac

,Check,High group avg risk %,Low group avg risk %,Expected,Result
0,High (>=3) vs zero prior inpatient visits,71.2,39.4,High > Low,PASS
1,Expired discharge (code 11) avg risk,0.6,N/A,< 5% (deceased),PASS
2,Long stay (>=10d) vs short stay (<=2d),39.8,33.7,Long > Short,PASS


## Output Contract Verification

Confirming the return shape matches the Phase 4 API contract exactly.

In [6]:
import json

sample_result = explain_patient(X.iloc[[0]], pipeline=pipeline)

# Simulate the Phase 4 /predict response shape
api_response = {
    "patient_id": "DEMO-001",
    "risk_percent": sample_result["risk_percent"],
    "risk_category": sample_result["risk_category"],
    "top_factors": sample_result["top_factors"],
    "follow_up_priority": sample_result["follow_up_priority"],
}

print(json.dumps(api_response, indent=2))

{
  "patient_id": "DEMO-001",
  "risk_percent": 23.0,
  "risk_category": "Low",
  "top_factors": [
    {
      "factor": "Number of medications prescribed",
      "direction": "decreases risk"
    },
    {
      "factor": "Discharged to home",
      "direction": "increases risk"
    },
    {
      "factor": "Number of prior inpatient visits",
      "direction": "decreases risk"
    }
  ],
  "follow_up_priority": "Low"
}


d:\DiabCare\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
